# Notebook 03 — Full Pipelines & Evaluation

Runs all three systems on the test split and computes evaluation metrics:
- **System A**: ColPali + MedGemma (RAG)
- **System B**: CLIP + MedGemma (RAG)
- **System C**: MedGemma Direct (no retrieval)

Metrics: BERTScore, ROUGE-L, RadGraph F1 (report gen) + BERTScore, BLEU-4, ROUGE-L (QA)

In [ ]:
!pip install -q 'transformers>=4.45.0' accelerate bitsandbytes
!pip install -q colpali-engine byaldi open-clip-torch faiss-gpu
!pip install -q bert-score rouge-score nltk
# RadGraph requires PhysioNet credentials; skip if not available
# !pip install -q radgraph

In [ ]:
import os, sys, json
import pandas as pd
from PIL import Image
from google.colab import drive, userdata
drive.mount('/content/drive')

DRIVE_ROOT  = '/content/drive/MyDrive/cxr_rag'
REPO_PATH   = '/content/cxr-rag-system'
sys.path.insert(0, REPO_PATH)

HF_TOKEN = userdata.get('HF_TOKEN')
IMAGES_DIR = '/content/openi/images'

In [ ]:
# ── Load reports corpus + test split ─────────────────────────────────────────
corpus_df = pd.read_csv(os.path.join(DRIVE_ROOT, 'reports_corpus.csv'))
test_df   = corpus_df[corpus_df['split'] == 'test'].head(100).reset_index(drop=True)
print(f'Evaluating on {len(test_df)} test studies')

# study_id → impression lookup (used to fetch context for retrieved images)
study_to_impression = dict(zip(corpus_df['study_id'], corpus_df['impression']))

In [ ]:
# ── Load MedGemma (shared across all systems) ─────────────────────────────────
from src.generation.medgemma_generator import MedGemmaGenerator
generator = MedGemmaGenerator(hf_token=HF_TOKEN, load_in_4bit=True)
print('MedGemma loaded')

In [ ]:
# ── Helper: run full RAG pipeline for one sample ──────────────────────────────
def run_report_pipeline(row, retriever, study_to_impression, k=3):
    image = Image.open(row['image_path']).convert('RGB')
    query = row.get('impression', 'chest x-ray findings')[:100]
    retrieved = retriever.search(query, k=k)
    context = [
        study_to_impression.get(
            os.path.splitext(os.path.basename(r['image_path']))[0], ''
        )
        for r in retrieved if r.get('image_path')
    ]
    context = [c for c in context if c][:k]
    return generator.generate_report(image, context_reports=context or None)

In [ ]:
# ── System A: ColPali + MedGemma ──────────────────────────────────────────────
import torch, gc

from src.retrieval.colpali_retriever import ColPaliRetriever
colpali = ColPaliRetriever.from_index(os.path.join(DRIVE_ROOT, 'colpali_index'))
colpali.load_path_map(os.path.join(DRIVE_ROOT, 'colpali_index'))

preds_A, refs_A = [], []
for _, row in test_df.iterrows():
    pred = run_report_pipeline(row, colpali, study_to_impression)
    preds_A.append(pred)
    refs_A.append(row['impression'])

del colpali; gc.collect(); torch.cuda.empty_cache()
print('System A done')

In [ ]:
# ── System B: CLIP + MedGemma ─────────────────────────────────────────────────
from src.retrieval.clip_retriever import CLIPRetriever
clip = CLIPRetriever()
clip.load_index(os.path.join(DRIVE_ROOT, 'clip_index'))

preds_B, refs_B = [], []
for _, row in test_df.iterrows():
    pred = run_report_pipeline(row, clip, study_to_impression)
    preds_B.append(pred)
    refs_B.append(row['impression'])

del clip; gc.collect(); torch.cuda.empty_cache()
print('System B done')

In [ ]:
# ── System C: MedGemma Direct (no retrieval) ──────────────────────────────────
preds_C, refs_C = [], []
for _, row in test_df.iterrows():
    image = Image.open(row['image_path']).convert('RGB')
    pred = generator.generate_report(image, context_reports=None)
    preds_C.append(pred)
    refs_C.append(row['impression'])
print('System C done')

In [ ]:
# ── Compute metrics ───────────────────────────────────────────────────────────
from src.evaluation.metrics import Evaluator
evaluator = Evaluator()

results = {}
for label, preds, refs in [
    ('ColPali + MedGemma (RAG)', preds_A, refs_A),
    ('CLIP + MedGemma (RAG)',    preds_B, refs_B),
    ('MedGemma Direct',         preds_C, refs_C),
]:
    results[label] = evaluator.evaluate_report_generation(preds, refs)
    print(f'{label}: {results[label]}')

In [ ]:
# ── Save results ──────────────────────────────────────────────────────────────
results_df = pd.DataFrame(results).T
print(results_df.to_markdown())
results_df.to_csv('/content/cxr-rag-system/evaluation/results.csv')
import shutil
shutil.copy('/content/cxr-rag-system/evaluation/results.csv',
            os.path.join(DRIVE_ROOT, 'results.csv'))

In [ ]:
# ── QA Evaluation ─────────────────────────────────────────────────────────────
qa_df = pd.read_json(os.path.join(DRIVE_ROOT, 'qa_dataset.jsonl'), lines=True)
qa_test = qa_df[qa_df['split'] == 'test'].head(50).reset_index(drop=True)

# Load ColPali for QA
from src.retrieval.colpali_retriever import ColPaliRetriever
colpali = ColPaliRetriever.from_index(os.path.join(DRIVE_ROOT, 'colpali_index'))
colpali.load_path_map(os.path.join(DRIVE_ROOT, 'colpali_index'))

qa_preds, qa_refs = [], []
for _, row in qa_test.iterrows():
    image = Image.open(row['image_path']).convert('RGB')
    retrieved = colpali.search(row['question'], k=3)
    context = [
        study_to_impression.get(
            os.path.splitext(os.path.basename(r['image_path']))[0], ''
        )
        for r in retrieved if r.get('image_path')
    ]
    context = [c for c in context if c][:3]
    pred = generator.answer_question(image, row['question'], context)
    qa_preds.append(pred)
    qa_refs.append(row['answer'])

qa_metrics = evaluator.evaluate_qa(qa_preds, qa_refs)
print('QA metrics (ColPali + MedGemma):', qa_metrics)